# Deploy & Serve LLMs in Production

## The Problem

Loading a model with `transformers` and calling `model.generate()` works for experimentation, but it's terrible for production:

- **Sequential** — one request at a time, everyone else waits
- **No batching** — GPU sits idle between tokens
- **Slow** — no kernel optimizations, no continuous batching

## Step 1: Measure Naive Inference

Load Llama 3 8B Instruct with plain `transformers` and time 5 sequential requests. Note the average — we'll compare to vLLM next.

In [ ]:
import torch
import time
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "/models/meta-llama--Meta-Llama-3-8B-Instruct"

print("Loading Llama 3 8B Instruct with transformers...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto",
)
print(f"Model loaded: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M params")
print(f"VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
test_prompts = [
    "What are the symptoms of diabetes?",
    "Explain how a transformer model works in 3 sentences.",
    "Write a Python function to calculate fibonacci numbers.",
    "What is the difference between TCP and UDP?",
    "Summarize the key ideas of reinforcement learning.",
]

print("Running 5 sequential requests...\n")
naive_times = []
for i, prompt in enumerate(test_prompts):
    messages = [{"role": "user", "content": prompt}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

    start = time.time()
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    elapsed = time.time() - start

    response = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    tokens = len(tokenizer.encode(response))
    naive_times.append(elapsed)
    print(f"  Request {i+1}: {elapsed:.1f}s ({tokens} tokens, {tokens/elapsed:.0f} tok/s)")

naive_avg = sum(naive_times) / len(naive_times)
print(f"\nAverage: {naive_avg:.1f}s per request")
print(f"Total for 5 requests: {sum(naive_times):.1f}s (sequential)")

### Free the GPU for vLLM

Naive loading took all the VRAM. Before launching vLLM, free it up.

> **Heads up:** In Jupyter, `del model` alone doesn't free GPU memory because IPython's output history (the `Out[]` dict and `_`, `__`, `___`) silently holds references to anything a cell returned. We must clear those too.

In [ ]:
import gc
from IPython import get_ipython

# Drop all direct references (tokenizer, model, tensors from the benchmark loop)
for _name in ['model', 'tokenizer', 'inputs', 'output', 'outputs', 'response']:
    if _name in globals():
        del globals()[_name]

# Critical: clear IPython's output history — it holds refs to previous cell outputs
ip = get_ipython()
if ip is not None:
    ip.run_line_magic('reset', '-f out')  # clears Out[] and _, __, ___

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

print(f"VRAM after cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB (allocated)")
print(f"VRAM reserved:       {torch.cuda.memory_reserved() / 1e9:.2f} GB")

In [ ]:
from preporato_labs import Lab
lab = Lab('deploy-serve-llms-jupyter')
lab.check(1)

## Step 2: Launch vLLM & Query It

**vLLM** is the most popular open-source LLM serving engine. It uses:
- **Continuous batching** — new requests join mid-generation
- **PagedAttention** — paged KV cache like an OS, no memory waste
- **Optimized CUDA kernels** — faster attention and sampling
- **OpenAI-compatible API** — drop-in replacement

### Launch the server

1. Open a **Terminal** from the JupyterLab Launcher (File → New → Terminal)
2. Run this command in the terminal:

```bash
python -m vllm.entrypoints.openai.api_server \
    --model /models/meta-llama--Meta-Llama-3-8B-Instruct \
    --dtype float16 \
    --max-model-len 2048 \
    --gpu-memory-utilization 0.85 \
    --port 9000
```

Wait for `Uvicorn running on http://0.0.0.0:9000` (takes 30-60s), then come back here.

In [ ]:
import httpx
import time

# Wait until vLLM is responsive
for attempt in range(60):
    try:
        r = httpx.get("http://localhost:9000/health", timeout=2)
        if r.status_code == 200:
            print(f"vLLM ready (after {attempt+1}s)")
            break
    except Exception:
        pass
    time.sleep(1)
else:
    print("vLLM not responding — make sure the server is running in the terminal")

In [ ]:
from openai import OpenAI

model_name = "/models/meta-llama--Meta-Llama-3-8B-Instruct"
client = OpenAI(base_url="http://localhost:9000/v1", api_key="not-needed")

def generate_vllm(prompt, max_tokens=200):
    start = time.time()
    completion = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=0.7,
    )
    response = completion.choices[0].message.content
    elapsed = time.time() - start
    return response, elapsed

response, t = generate_vllm("What is the capital of France?")
print(f"Response ({t:.1f}s): {response[:300]}")

In [ ]:
prompts = [
    "What are the symptoms of diabetes?",
    "Explain how a transformer model works in 3 sentences.",
    "Write a Python function to calculate fibonacci numbers.",
    "What is the difference between TCP and UDP?",
    "Summarize the key ideas of reinforcement learning.",
]

print("Running 5 sequential requests via vLLM...\n")
vllm_times = []
for i, prompt in enumerate(prompts):
    r, t = generate_vllm(prompt)
    vllm_times.append(t)
    print(f"  Request {i+1}: {t:.1f}s")

vllm_avg = sum(vllm_times) / len(vllm_times)
print(f"\nAverage (vLLM): {vllm_avg:.1f}s per request")
if 'naive_avg' in globals():
    print(f"Speedup over naive: {naive_avg/vllm_avg:.1f}x")

### Visualize: naive vs vLLM

The numbers tell the story, but a chart makes it instant. Let's plot per-request latency side by side:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
%matplotlib inline

fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(1, 6)
width = 0.38
ax.bar(x - width/2, naive_times, width, label=f'Naive transformers (avg {naive_avg:.1f}s)', color='#e74c3c')
ax.bar(x + width/2, vllm_times, width, label=f'vLLM (avg {vllm_avg:.1f}s)', color='#2ecc71')
ax.set_xlabel('Request #')
ax.set_ylabel('Latency (seconds)')
ax.set_title(f'Per-request latency: naive vs vLLM  —  {naive_avg/vllm_avg:.1f}x speedup on average')
ax.set_xticks(x)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from preporato_labs import Lab
lab = Lab('deploy-serve-llms-jupyter')
lab.check(2)

## Step 3: Benchmark Under Load

The real test isn't single-request speed — it's **concurrent requests**. This is where vLLM's continuous batching shines.

### What is continuous batching?

Traditional batching waits for a batch to fill, processes it, then waits again. Continuous batching adds new requests **mid-generation** — while token 5 of request A is being generated, request B starts its token 1.

> **NCP-GENL exam note:** Continuous batching vs static batching is a common exam topic.

In [ ]:
import asyncio
from openai import AsyncOpenAI

async def benchmark_concurrent(prompts, max_tokens=150):
    aclient = AsyncOpenAI(base_url="http://localhost:9000/v1", api_key="not-needed")

    async def single_request(prompt):
        start = time.time()
        completion = await aclient.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
            temperature=0.7,
        )
        elapsed = time.time() - start
        tokens = completion.usage.completion_tokens
        return {"prompt": prompt[:50], "latency": elapsed, "tokens": tokens}

    return list(await asyncio.gather(*[single_request(p) for p in prompts]))

concurrent_prompts = [
    "What causes high blood pressure?",
    "Explain quantum computing simply.",
    "Write a haiku about machine learning.",
    "What are the SOLID principles?",
    "How does photosynthesis work?",
    "What is the difference between SQL and NoSQL?",
    "Explain the attention mechanism in transformers.",
    "What are the benefits of exercise?",
    "How does HTTPS encryption work?",
    "Summarize the history of artificial intelligence.",
]

print("Sending 10 concurrent requests to vLLM...\n")
total_start = time.time()
concurrent_results = await benchmark_concurrent(concurrent_prompts)
concurrent_total = time.time() - total_start

for r in concurrent_results:
    print(f"  {r['prompt']:<50}  {r['latency']:.1f}s  ({r['tokens']} tokens)")

concurrent_avg = sum(r['latency'] for r in concurrent_results) / len(concurrent_results)
concurrent_tokens = sum(r['tokens'] for r in concurrent_results)

print(f"\n--- 10 Concurrent Requests ---")
print(f"Total wall time:    {concurrent_total:.1f}s")
print(f"Average latency:    {concurrent_avg:.1f}s per request")
print(f"Total tokens:       {concurrent_tokens}")
print(f"Throughput:          {concurrent_tokens/concurrent_total:.0f} tokens/sec")
print(f"\n10 naive sequential requests would take ~50s.")
print(f"vLLM handled all 10 in {concurrent_total:.1f}s — continuous batching in action.")

### Visualize: per-request latency under load

Notice how the latencies are clustered tight together (continuous batching) rather than serialized (naive would look like a staircase):

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
latencies = [r['latency'] for r in concurrent_results]
labels = [f"{r['prompt'][:30]}..." for r in concurrent_results]
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(latencies)))
ax.barh(range(len(latencies)), latencies, color=colors)
ax.set_yticks(range(len(latencies)))
ax.set_yticklabels(labels, fontsize=8)
ax.set_xlabel('Latency (seconds)')
ax.set_title(f'10 concurrent requests — wall time {concurrent_total:.1f}s, avg latency {concurrent_avg:.1f}s')
ax.axvline(concurrent_total, color='red', linestyle='--', linewidth=1, label=f'Wall time: {concurrent_total:.1f}s')
ax.legend()
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nAll {len(latencies)} requests completed in roughly the same window — that's continuous batching.")
print(f"Without it, wall time would be ~{sum(latencies):.0f}s (serial sum of latencies).")

In [ ]:
from preporato_labs import Lab
lab = Lab('deploy-serve-llms-jupyter')
lab.check(3)

## Step 4: Tune vLLM Settings

vLLM has settings that directly affect performance:

| Setting | What it controls | Tradeoff |
|---------|-----------------|----------|
| `--max-model-len` | Max sequence length (input + output) | Lower = more concurrent requests fit in VRAM |
| `--gpu-memory-utilization` | Fraction of VRAM for KV cache | Higher = more requests, less headroom |
| `--dtype` | Model precision | float16 fastest on modern GPUs |
| `--quantization` | AWQ/GPTQ/SqueezeLLM | Smaller model = more KV cache space |

### Experiment

1. In your **Terminal** tab, press `Ctrl+C` to stop vLLM
2. Restart with a shorter context:

```bash
python -m vllm.entrypoints.openai.api_server \
    --model /models/meta-llama--Meta-Llama-3-8B-Instruct \
    --dtype float16 \
    --max-model-len 512 \
    --gpu-memory-utilization 0.90 \
    --port 9000
```

3. Wait for it to start, then run the cells below.

In [ ]:
# Wait for vLLM to come back up on the new settings
for attempt in range(90):
    try:
        r = httpx.get("http://localhost:9000/health", timeout=2)
        if r.status_code == 200:
            print(f"vLLM ready (after {attempt+1}s)")
            break
    except Exception:
        pass
    time.sleep(1)

metrics = httpx.get("http://localhost:9000/metrics", timeout=5).text
print("\nKey vLLM metrics:")
for line in metrics.split('\n'):
    if any(k in line for k in ['gpu_cache_usage', 'num_requests', 'avg_generation_throughput']) and not line.startswith('#'):
        print(f"  {line}")

In [ ]:
tune_prompts = [
    "What causes high blood pressure?",
    "Explain quantum computing simply.",
    "Write a haiku about machine learning.",
    "What are the SOLID principles?",
    "How does photosynthesis work?",
    "What is the difference between SQL and NoSQL?",
    "Explain the attention mechanism in transformers.",
    "What are the benefits of exercise?",
    "How does HTTPS encryption work?",
    "Summarize the history of AI.",
]

print("10 concurrent requests with optimized settings...\n")
tune_start = time.time()
tune_results = await benchmark_concurrent(tune_prompts, max_tokens=100)
tune_total = time.time() - tune_start

tune_avg = sum(r['latency'] for r in tune_results) / len(tune_results)
tune_tokens = sum(r['tokens'] for r in tune_results)

print(f"Total wall time:  {tune_total:.1f}s")
print(f"Average latency:  {tune_avg:.1f}s")
print(f"Throughput:        {tune_tokens/tune_total:.0f} tokens/sec")
if 'concurrent_total' in globals():
    print(f"\nPrevious (max_model_len=2048): {concurrent_total:.1f}s total")
    print(f"New (max_model_len=512):        {tune_total:.1f}s total")

### Visualize: throughput by `max_model_len`

Shorter context = smaller KV cache per request = more requests fit in VRAM = higher throughput:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))

configs = ['max_model_len=2048', 'max_model_len=512']
throughputs = [concurrent_tokens/concurrent_total, tune_tokens/tune_total]
wall_times = [concurrent_total, tune_total]

colors = ['#3498db', '#27ae60']
axes[0].bar(configs, throughputs, color=colors)
axes[0].set_ylabel('Tokens / second')
axes[0].set_title('Throughput')
for i, v in enumerate(throughputs):
    axes[0].text(i, v, f'{v:.0f}', ha='center', va='bottom', fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(configs, wall_times, color=colors)
axes[1].set_ylabel('Wall time (seconds)')
axes[1].set_title('10 concurrent requests — total time')
for i, v in enumerate(wall_times):
    axes[1].text(i, v, f'{v:.1f}s', ha='center', va='bottom', fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nThroughput change: {(throughputs[1]/throughputs[0] - 1)*100:+.0f}% with max_model_len=512")

In [ ]:
from preporato_labs import Lab
lab = Lab('deploy-serve-llms-jupyter')
lab.check(4)

## Step 5: Production Patterns

### Streaming

Users expect to see tokens appear as they're generated. vLLM supports streaming via the standard OpenAI `stream=True` parameter.

The cell below demonstrates streaming and measures **Time to First Token (TTFT)** — the key UX metric for chat applications.

In [ ]:
streamed_text = ""
token_times = []
start = time.time()
ttft = None

stream = client.chat.completions.create(
    model=model_name,
    messages=[{"role": "user", "content": "Explain PagedAttention in vLLM and why it matters for serving."}],
    max_tokens=300,
    temperature=0.7,
    stream=True,
)

print("Streaming response (watch tokens appear):\n")
for chunk in stream:
    if chunk.choices[0].delta.content:
        token = chunk.choices[0].delta.content
        now = time.time() - start
        token_times.append(now)
        if ttft is None:
            ttft = now
        streamed_text += token
        print(token, end="", flush=True)

stream_total = time.time() - start
print(f"\n\n--- Streaming Stats ---")
print(f"TTFT (time to first token): {ttft*1000:.0f}ms")
print(f"Total generation time:       {stream_total:.1f}s")
print(f"Tokens generated:            {len(token_times)}")
print(f"\nTTFT is what users feel — {ttft*1000:.0f}ms means near-instant response start.")

### Visualize: token arrival timeline

Non-streaming responses feel slow because the user waits for everything. Streaming trades small marginal latency for a **near-instant perceived response start**:

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 3.5))
token_indices = np.arange(1, len(token_times) + 1)
ax.plot(token_times, token_indices, color='#2ecc71', linewidth=2)
ax.axvline(ttft, color='#e67e22', linestyle='--', label=f'TTFT: {ttft*1000:.0f}ms')
ax.axvline(stream_total, color='#3498db', linestyle='--', label=f'Total: {stream_total:.1f}s')
ax.fill_betweenx(token_indices, 0, token_times, alpha=0.15, color='#2ecc71')
ax.set_xlabel('Time since request (seconds)')
ax.set_ylabel('Tokens received')
ax.set_title('Streaming response — user sees tokens appear continuously')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

tok_per_sec = len(token_times) / stream_total
print(f"\nAverage token rate: {tok_per_sec:.0f} tokens/sec after first token")
print(f"Without streaming, user would wait {stream_total:.1f}s seeing nothing.")
print(f"With streaming, first response visible after only {ttft*1000:.0f}ms.")

### When to use what

| Engine | Best for | Not ideal for |
|--------|----------|---------------|
| **vLLM** | LLM serving, high throughput | Non-LLM models (vision, audio) |
| **Triton** | Multi-model serving, ensemble pipelines | Simple single-LLM deployments |
| **TGI** | Quick HuggingFace model deployment | Maximum throughput at scale |
| **Ollama** | Local development, easy setup | Production deployments |
| **TensorRT-LLM** | Maximum single-request speed | Flexibility, non-NVIDIA hardware |

**In practice:** Most production LLM APIs use vLLM behind a load balancer. Triton is used when you need to serve multiple model types (LLM + embedding + reranker) in one system.

### What you've learned

1. Naive inference is easy but slow — sequential, no batching
2. vLLM serves the same model much faster using optimized CUDA kernels and continuous batching
3. Concurrent requests show the real difference — continuous batching keeps the GPU busy
4. Tuning `max_model_len` and `gpu_memory_utilization` directly affects capacity
5. Streaming reduces perceived latency with instant TTFT
6. vLLM for LLM serving, Triton for multi-model, TensorRT-LLM for max speed

> These are the exact concepts tested on the **NCP-GENL exam** (Domain 1: Deployment & Inference — 26%).

In [ ]:
from preporato_labs import Lab
lab = Lab('deploy-serve-llms-jupyter')
lab.check(5)